# Compact MobileNetV4 student — KITTI GT baseline
**Latest notebook: COMPACT-MNV4-KITTI-GT-2026-10-09-r1**

Purpose: train the 1.48M-parameter MobileNetV4 Conv Small + narrow pyramid + shared nine-head CNN that passed the untrained iPhone CPU speed screen. This is **not the A2 transformer** and is **not distillation yet**. We establish the smaller student's own accuracy first.

The phone screen measured model-only p95 27.9 ms and 32.9 FPS with random weights. It did not measure detection accuracy, camera processing or the 3D decoder. Training uses RGB/ImageNet normalization to match the ImageNet backbone warm start; the older untrained phone fixture was BGR. A trained export must include the new RGB contract and be rechecked on the phone. The architecture, resolution and head shapes stay unchanged.

Run sections **1–6 in order** on a Colab GPU. No M62/M67 virtual environment, CUDA compilation or external teacher is needed. KITTI images, labels and calibration plus the Chen train/val split must already be on Drive. Full training resumes from the last completed epoch. Keep the same RUN_ID for an unchanged run; use a new RUN_ID after changing code/config/data. Never delete old results to bypass a check.

Deliverables: data/class counts, finite-gradient smoke, a train-only tiny learnability check, resumable checkpoints, validation AP_R40 and a small review ZIP. The first baseline keeps the natural Vehicle/Pedestrian distribution with no added class weighting. Later distillation will be compared against this control, not assumed to help.

## 1. Mount Drive, set paths and define all helpers
This cell contains every import and helper used below. You do not need to copy cells from another notebook.

In [ ]:
# COMPACT-MNV4-KITTI-GT-2026-10-09-r1 — self-contained training baseline
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
from collections import deque
import json, os, shlex, subprocess, sys

REVISION = 'COMPACT-MNV4-KITTI-GT-2026-10-09-r1'
RUN_ID = 'compact_mnv4_gt_r1'
RUN_FULL_TRAINING = True  # Full run starts only after smoke + tiny checks pass.
MOBILE_REPO = Path('/content/mobile_adas3d')
DATASET = Path('/content/kitti_compact_mnv4')
SPLITS = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
OUTPUT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/compact_mnv4_gt') / RUN_ID
CONFIG = MOBILE_REPO / 'configs/compact_mnv4_kitti_gt_r1.yaml'
SCRIPT = MOBILE_REPO / 'scripts/train_compact_kitti.py'
LOGS = OUTPUT / 'colab_logs'
LOGS.mkdir(parents=True, exist_ok=True)
PYTHON = Path(sys.executable)
READY = False

def run_logged(command, name, cwd=None, allow_failure=False):
    command = [str(item) for item in command]
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    log_path = LOGS / f'{name}_{stamp}.log'
    print('+', shlex.join(command), flush=True)
    print('Durable log:', log_path, flush=True)
    tail = deque(maxlen=40)
    env = os.environ.copy(); env['PYTHONUNBUFFERED'] = '1'
    with log_path.open('w') as log:
        log.write(shlex.join(command) + '\n'); log.flush()
        process = subprocess.Popen(command, cwd=str(cwd) if cwd else None, env=env,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush(); tail.append(line.rstrip())
        except BaseException:
            process.terminate()
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
            raise
        finally:
            process.stdout.close()
        code = process.wait(); log.write(f'\nEXIT CODE: {code}\n')
    if code and not allow_failure:
        raise RuntimeError(f'Exit {code}; full log={log_path}\n' + '\n'.join(tail))
    return code, log_path

def stage(name, *extra):
    if not READY: raise RuntimeError('Run sections 1 and 2 first')
    return run_logged([PYTHON, '-u', SCRIPT, name, '--config', CONFIG,
                       '--dataset-root', DATASET, '--split-dir', SPLITS,
                       '--output-dir', OUTPUT, '--device', 'cuda', *extra],
                      name, MOBILE_REPO)

print(REVISION, '\nOutput:', OUTPUT)
print('After a session reset: rerun 1–6; completed tiny check and epoch checkpoints can be reused.')

## 2. Update this project, verify GPU and restore data
No upstream RTM3D clone or native extension is required. This cell does not replace Colab's Torch/CUDA packages. It preserves a modified checkout rather than resetting it.

In [ ]:
READY = False
if not MOBILE_REPO.exists():
    run_logged(['git', 'clone', '--branch', 'main', 'https://github.com/Ali-RT/mobile_adas3d.git', MOBILE_REPO], 'clone_mobile')
dirty = subprocess.check_output(['git','status','--porcelain','--untracked-files=no'], cwd=MOBILE_REPO, text=True).strip()
if dirty: raise RuntimeError('Project checkout has tracked edits; preserve them and use a clean main checkout.\n' + dirty)
branch = subprocess.check_output(['git','branch','--show-current'], cwd=MOBILE_REPO, text=True).strip()
if branch != 'main': raise RuntimeError(f'Use project main, not {branch!r}')
run_logged(['git','pull','--ff-only','origin','main'], 'update_mobile', MOBILE_REPO)
for required in (SCRIPT, CONFIG, MOBILE_REPO/'models/compact_monocular3d.py', MOBILE_REPO/'data/compact_kitti.py'):
    if not required.is_file(): raise FileNotFoundError(f'{required}: this notebook needs the published compact training revision on main')
run_logged([PYTHON,'-m','pip','install','-q','timm==1.0.27','pyyaml','opencv-python-headless','numpy'], 'dependencies')
run_logged(['nvidia-smi'], 'gpu_allocation')
probe = "import torch,timm; assert torch.cuda.is_available(); assert timm.__version__ == '1.0.27'; print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0)); x=torch.ones(8,device='cuda'); print(float((x*x).sum()))"
run_logged([PYTHON,'-c',probe], 'cuda_probe')
# Restore only missing local links, validating all Chen train/val paths.
candidates = [Path('/content/drive/MyDrive/datasets/kitti'), Path('/content/kitti'),
              Path('/content/kitti_m67'), Path('/content/kitti_m66'),
              Path('/content/monodetr_kitti_a2')]
command = [PYTHON, MOBILE_REPO/'scripts/restore_m63_data.py', '--dataset', DATASET, '--splits', SPLITS]
for candidate in candidates: command += ['--candidate', candidate]
run_logged(command, 'restore_kitti', MOBILE_REPO)
READY = True
print('Setup ready. No M62/M67/M63 runtime or compiled CUDA operation was loaded.')

## 3. Tests, taxonomy counts and real GPU gradient smoke
GT-only means no external teacher. ImageNet backbone initialization is used; all new neck/heads learn from KITTI labels. The smoke verifies two optimizer steps, not model accuracy.

In [ ]:
run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_compact_kitti_training.py','-v'], 'training_tests', MOBILE_REPO)
run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_compact_kitti_notebook.py','-v'], 'notebook_tests', MOBILE_REPO)
stage('preflight')
stage('smoke')
smoke = json.loads((OUTPUT/'smoke.json').read_text())
assert smoke['passed'] and smoke['finite_gradients_all_parameters']
print('GPU smoke passed. Next: train-only tiny learnability check.')

## 4. Tiny learnability check — 16 training images, 400 steps
This disposable check must reduce the same-batch total loss by at least 20%. That is an engineering gate, **not AP accuracy or successful 3D overfitting**. Its weights are never used to initialize full training. Stop and return the review ZIP if it fails.

In [ ]:
tiny_path = OUTPUT/'tiny.json'
run_identity = json.loads((OUTPUT/'training_manifest.json').read_text())['identity_sha256']
saved_tiny = json.loads(tiny_path.read_text()) if tiny_path.exists() else {}
if saved_tiny.get('passed') and saved_tiny.get('identity_sha256') == run_identity:
    print('Reusing passed tiny learnability check for this unchanged run.')
else:
    stage('tiny', '--steps', '400')
tiny = json.loads(tiny_path.read_text())
assert tiny['passed'] and tiny['finite_gradients_all_parameters']
print({k:tiny[k] for k in ('steps','initial_same_batch_loss','final_same_batch_loss','loss_ratio','elapsed_seconds')})

## 5. Train one GT-only control and evaluate every 10 epochs
140 epochs, batch size 4, FP32, AdamW, augmentation, natural class distribution, no teacher/class-weight sweep. Validation uses all 3,769 Chen frames and the same Vehicle/Pedestrian product AP_R40 evaluator as A2. Atomic checkpoints save every completed epoch; numbered checkpoints every 5 epochs. Resume is epoch-boundary, not mid-batch. Leave RUN_ID unchanged after a disconnect. Full training starts from ImageNet + fresh neck/heads, not the tiny run.

In [ ]:
if not json.loads((OUTPUT/'tiny.json').read_text()).get('passed'): raise RuntimeError('Tiny check failed; do not train')
if RUN_FULL_TRAINING:
    extra = ['--resume'] if (OUTPUT/'checkpoints/latest.pt').is_file() else []
    stage('train', *extra)
    selection = json.loads((OUTPUT/'selection.json').read_text())
    print(json.dumps(selection, indent=2))
else:
    print('Full training disabled. Return the smoke/tiny result ZIP first.')

## 6. Package the results for review
You can run this section after a failure, too. The ZIP includes reports and durable logs, **not large checkpoints**; checkpoints stay on Drive. Return the ZIP before starting distillation. A trained export and full phone pipeline test are separate next steps.

In [ ]:
stage('bundle')
BUNDLE = OUTPUT/'compact_mnv4_gt_results.zip'
print('Review bundle:', BUNDLE)
if (OUTPUT/'selection.json').is_file():
    selected = json.loads((OUTPUT/'selection.json').read_text())
    print('Selected trained checkpoint:', selected['checkpoint'])
from google.colab import files
files.download(str(BUNDLE))